# 🧪 Sesión 18 · Rate Limiting, Throttling y Prevención de Abuso

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo evitas que un script sature tu API?

---

## 🖥️ Así se verá el resultado final

```http
GET /api/recurso   -> 200 OK   (1.ª petición)
GET /api/recurso   -> 200 OK   (2.ª petición)
GET /api/recurso   -> 200 OK   (3.ª petición)
GET /api/recurso   -> 429 Too Many Requests
Retry-After: 60
{"error":"límite de peticiones alcanzado"}
```

_El límite protege el servicio para todos; no es un castigo al cliente._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Implementar un límite de peticiones por cliente.
- Responder 429 con Retry-After.
- Explicar la diferencia entre limitar y bloquear.

## ✅ Criterios de aceptación

El verificador ejecutará estos casos. Cada uno debe quedar en verde:

- [ ] Primera petición permitida → HTTP 200
- [ ] Segunda petición permitida → HTTP 200
- [ ] Tercera petición permitida → HTTP 200
- [ ] La cuarta petición se limita → HTTP 429

## 🧩 El reto

Implementa un límite de tres peticiones por minuto por cliente y responde 429 cuando se exceda.

**Requisitos del contrato**

- GET /api/recurso permite tres peticiones por cliente en una ventana de 60 segundos.
- La cuarta petición responde 429.
- La respuesta 429 incluye Retry-After.
- Las tres primeras responden 200 con un JSON simple.
- El contador se reinicia al iniciar la aplicación.

> **Pista conceptual:** El cliente se identifica por IP o por token. Un contador global castiga a todos; un contador por cliente protege a todos.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion18.md` con el cálculo de la ventana.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `app.py`


### 📄 `app.py`

Archivo principal de la implementación.

In [2]:

%%writefile app.py

from fastapi import FastAPI, Request
from fastapi.responses import JSONResponse
import time
import math

app = FastAPI(title="API con Rate Limiting")

# Configuración del límite
LIMITE_PETICIONES = 3
VENTANA_SEGUNDOS = 60

# Almacenar las peticiones de cada cliente
clientes = {}


# Reiniciar el contador al iniciar la aplicación
@app.on_event("startup")
def reiniciar_contadores():
    clientes.clear()


# Ruta protegida con límite de peticiones
@app.get("/api/recurso")
def obtener_recurso(request: Request):

    # Identificar al cliente por su dirección IP
    ip_cliente = request.client.host if request.client else "desconocido"

    tiempo_actual = time.monotonic()

    # Crear contador para un cliente nuevo
    if ip_cliente not in clientes:
        clientes[ip_cliente] = {
            "inicio": tiempo_actual,
            "peticiones": 0
        }

    cliente = clientes[ip_cliente]

    # Reiniciar la ventana después de 60 segundos
    if tiempo_actual - cliente["inicio"] >= VENTANA_SEGUNDOS:
        cliente["inicio"] = tiempo_actual
        cliente["peticiones"] = 0

    # Comprobar si el cliente superó el límite
    if cliente["peticiones"] >= LIMITE_PETICIONES:

        tiempo_restante = VENTANA_SEGUNDOS - (
            tiempo_actual - cliente["inicio"]
        )

        retry_after = max(1, math.ceil(tiempo_restante))

        return JSONResponse(
            status_code=429,
            content={
                "error": "límite de peticiones alcanzado"
            },
            headers={
                "Retry-After": str(retry_after)
            }
        )

    # Registrar la petición permitida
    cliente["peticiones"] += 1

    return {
        "mensaje": "Petición permitida",
        "peticiones": cliente["peticiones"]
    }


Writing app.py


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** |Python 3 y FastAPI en Google Colab. Se utilizará un contador en memoria para controlar las peticiones.|
| **Rol** | Desarrollador backend especializado en APIs REST, seguridad y control de tráfico.|
| **Especificación** | Crear la ruta GET /api/recurso que permita tres peticiones por cliente durante 60 segundos. La cuarta debe responder 429 e incluir el encabezado Retry-After. |
| **Output** | Código completo en app.py, con control de peticiones por IP, respuestas JSON y manejo correcto del tiempo de espera.|

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [3]:
from verificar import probar_http

casos = [{'descripcion': 'Primera petición permitida', 'metodo': 'GET', 'ruta': '/api/recurso', 'estado': 200},
 {'descripcion': 'Segunda petición permitida', 'metodo': 'GET', 'ruta': '/api/recurso', 'estado': 200},
 {'descripcion': 'Tercera petición permitida', 'metodo': 'GET', 'ruta': '/api/recurso', 'estado': 200},
 {'descripcion': 'La cuarta petición se limita',
  'metodo': 'GET',
  'ruta': '/api/recurso',
  'estado': 429,
  'headers_contiene': {'retry-after': ''},
  'json_contiene': {'error': 'límite de peticiones alcanzado'}}]

probar_http(casos, archivo="app.py", limpiar=[], env={})


VERIFICACIÓN DEL RESULTADO · 4 casos · app.py
✅ 01. Primera petición permitida
✅ 02. Segunda petición permitida
✅ 03. Tercera petición permitida
✅ 04. La cuarta petición se limita
------------------------------------------------------------------------------
RESULTADO: 4/4 casos correctos. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion18.md` a `auditorias/auditoria_sesion18.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   Lo que más me costó fue controlar correctamente el tiempo de las peticiones, porque necesitaba que el contador se reiniciara después de 60 segundos y no antes. También tuve que cuidar que cada cliente tuviera su propio contador para no afectar a los demás usuarios.
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   Decidí utilizar un diccionario para guardar las peticiones de cada dirección IP y utilizar la función time.monotonic() para medir el tiempo transcurrido. Esto me permitió organizar mejor los contadores y controlar la ventana de tiempo.
   - ¿Qué error detectó la verificación que tú no habías previsto?
   Un detalle que tuve que cuidar fue que la cuarta petición no solamente debía devolver el código 429, sino que también debía incluir el encabezado Retry-After. Así el cliente puede saber cuánto tiempo debe esperar antes de volver a realizar una solicitud.
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
